In [4]:
import pandas as pd
import numpy as np
import os

# PHASE 1: VCF EXTRACTION & PARSING PIPELINE

def extract_info(info_text):
    """
    Parses the semi-structured INFO column of a VCF file into a dictionary.
    """
    if pd.isna(info_text): 
        return {}
    
    info_dict = {}
    for item in str(info_text).split(';'):
        if '=' in item:
            key, value = item.split('=')
            info_dict[key] = value
            
    return info_dict

def run_vcf_pipeline(file_path, rows_to_read=None):
    """
    Reads, cleans, and standardizes VCF data dynamically.
    """
    print(f"[*] Initializing data extraction from: {file_path}")
    
    # Check if the file exists in the directory
    if not os.path.exists(file_path):
        print(f"❌ Error: File not found at {file_path}. Please verify the path.")
        return None
        
    header_line = None
    
    # Dynamically located the exact row where the dataset begins
    try:
        with open(file_path, 'r') as file:
            for line_index, line in enumerate(file):
                if line.startswith('#CHROM'):
                    header_line = line_index
                    break
                    
        if header_line is None:
            raise ValueError("Corrupted VCF: '#CHROM' header line is missing.")
            
        # Load the VCF into a Pandas DataFrame
        df = pd.read_csv(file_path, sep='\t', skiprows=header_line, nrows=rows_to_read)
        
    except Exception as e:
        print(f"❌ Pipeline failed during data ingestion: {e}")
        return None
    
    # DATA SCRUBBING & TRANSFORMATION
    # Replaced VCF specific missing values ('.') with standard NumPy NaN
    df = df.replace('.', np.nan)
    
    if 'QUAL' in df.columns:
        df['QUAL'] = pd.to_numeric(df['QUAL'], errors='coerce')
    
    # Parse the complex INFO column into distinct features
    if 'INFO' in df.columns:
        info_df = pd.DataFrame(df['INFO'].apply(extract_info).tolist())
        df = pd.concat([df, info_df], axis=1).drop('INFO', axis=1)
    
    if 'DP' in df.columns:
        df['DP'] = pd.to_numeric(df['DP'], errors='coerce')
        
    print("[+] VCF Parsing Completed Successfully!\n")
    return df


# EXECUTION: CLINVAR DATASET

print("--- CLINVAR DATA ANALYSIS ---")
clinvar_result = run_vcf_pipeline('./data/clinvar.vcf', rows_to_read=10000)

# Filtered for Pathogenic Mutations
if clinvar_result is not None and 'CLNSIG' in clinvar_result.columns:
    
    # Isolated rows where Clinical Significance indicates pathogenicity
    pathogenic_mutations = clinvar_result[clinvar_result['CLNSIG'].str.contains('Pathogenic', na=False)]
    
    # Safely selected required columns for downstream analysis
    expected_cols = ['#CHROM', 'POS', 'GENEINFO', 'CLNDN', 'CLNSIG']
    safe_cols = [col for col in expected_cols if col in clinvar_result.columns]
    
    final_view = pathogenic_mutations[safe_cols]
    
    print(f"[*] Successfully isolated {len(final_view)} Pathogenic Mutations.\n")
    print(final_view.head())
    
    # Exported the raw pathogenic data for Phase 2 (Data Cleaning)
    output_filename = './data/114_pathogenic_mutations_raw.csv'
    final_view.to_csv(output_filename, index=False)
    print(f"\n[+] Data successfully exported to {output_filename}")
    print("[+] Phase 1 (Extraction) Complete! 🚀")
else:
    print("[-] Target column 'CLNSIG' was not found in this dataset.")

--- CLINVAR DATA ANALYSIS ---
[*] Initializing data extraction from: ./data/clinvar.vcf


C:\Users\Sarthak Shukla\AppData\Local\Temp\ipykernel_4124\3162966788.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('.', np.nan)


[+] VCF Parsing Completed Successfully!

[*] Successfully isolated 114 Pathogenic Mutations.

      #CHROM      POS                 GENEINFO  \
1026       1   943995            SAMD11:148398   
1739       1   976611  AGRN:375790|PERM1:84808   
1830       1  1013983               ISG15:9636   
1872       1  1014143               ISG15:9636   
1915       1  1014316               ISG15:9636   

                                                  CLNDN  \
1026                                       not_provided   
1739                   Congenital_myasthenic_syndrome_8   
1830  Mendelian_susceptibility_to_mycobacterial_dise...   
1872  Mendelian_susceptibility_to_mycobacterial_dise...   
1915  Mendelian_susceptibility_to_mycobacterial_dise...   

                            CLNSIG  
1026                    Pathogenic  
1739                    Pathogenic  
1830  Pathogenic/Likely_pathogenic  
1872                    Pathogenic  
1915                    Pathogenic  

[+] Data successfully expor